# Food Landscape GA

Generate levels by evolving a population on a "food landscape" defined by the
original level distribution (KDE). Each individual consumes food at its location
(a negative Gaussian), so the population self-organizes to match the KDE shape.

`fitness(i) = kde_density(feat_i) - crowding_penalty(feat_i, population)`

Includes an animation of the population spreading across the feature space over generations.

In [ ]:
import random
import numpy as np
import mummymaze_rust
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from scipy.stats import gaussian_kde
from IPython.display import HTML

MAZE_DIR = Path("../mazes")

# ============================================================
# CONFIGURATION
# ============================================================
GRID_SIZE = 8  # single grid size for this experiment

# GA parameters
POP_SIZE = 128
GENERATIONS = 100
TOURNAMENT_K = 3
ELITE_FRAC = 0.1
MUTATION_RATE = 0.8
CROSSOVER_MODES = ["swap_entities", "wall_patch", "region"]
W_WALL = 5.0
W_MOVE_ENTITY = 3.0
W_MOVE_PLAYER = 2.0
W_MOVE_EXIT = 1.0
EXTRA_WALL_PROB = 0.3

# Food landscape parameters
CROWDING_RADIUS = 0.5   # radius of each individual's "consumption" Gaussian (in normalized space)
CROWDING_STRENGTH = 1.0 # how much each neighbor depletes food

# Entity bin to generate for (None = all levels regardless of composition)
ENTITY_BIN = None  # e.g. (True, False, 0, "none") for 2 mummies, no extras

# Log-transform N states before KDE fitting
LOG_DIMS = [1]

SEED = 42
SNAPSHOT_EVERY = 1  # save population snapshot every N generations for animation

## 1. Parse originals, compute features, fit KDE

In [ ]:
FEATURE_NAMES = ["BFS moves", "N states", "Log win prob", "Wall count"]


def wall_count(level):
    d = level.to_dict()
    gs = d["grid_size"]
    h, v = d["h_walls"], d["v_walls"]
    h_interior = sum(h[i] for i in range(gs, gs * gs))
    v_interior = sum(v[r * (gs + 1) + c] for r in range(gs) for c in range(1, gs))
    return h_interior + v_interior


def level_features(eval_result, level):
    return np.array([
        eval_result["bfs_moves"],
        eval_result["n_states"],
        eval_result["log_win_prob"],
        wall_count(level),
    ], dtype=np.float64)


def transform_features(feats):
    out = feats.copy()
    for d in LOG_DIMS:
        out[:, d] = np.log1p(np.maximum(out[:, d], 0))
    return out


def untransform_features(feats):
    out = feats.copy()
    for d in LOG_DIMS:
        out[:, d] = np.expm1(out[:, d])
    return out


# Parse and evaluate
print("Parsing and evaluating original levels...")
all_levels = []
for dat_file in sorted(MAZE_DIR.glob("B-*.dat")):
    try:
        levels = mummymaze_rust.parse_file(str(dat_file))
    except Exception:
        continue
    for lev in levels:
        if lev.to_dict()["grid_size"] == GRID_SIZE:
            all_levels.append(lev)

print(f"  {len(all_levels)} levels for gs={GRID_SIZE}")

results = mummymaze_rust.ga_evaluate_batch(all_levels)
orig_features_raw = np.array([level_features(r, r["level"]) for r in results])
print(f"  {len(orig_features_raw)} solvable")

# Transform and normalize for KDE
orig_transformed = transform_features(orig_features_raw)
t_mean = orig_transformed.mean(axis=0)
t_std = orig_transformed.std(axis=0)
t_std[t_std < 1e-8] = 1.0
orig_normed = (orig_transformed - t_mean) / t_std

# Fit KDE on normalized+transformed features
kde = gaussian_kde(orig_normed.T)
print(f"  KDE bandwidth factor: {kde.factor:.4f}")

# Also compute raw feature scales for fitness normalization
raw_std = orig_features_raw.std(axis=0)
raw_std[raw_std < 1e-8] = 1.0

## 2. Food landscape GA

In [ ]:
def to_normed(raw_feats):
    """Raw features -> transformed+normalized space (for KDE/crowding)."""
    t = transform_features(raw_feats.reshape(1, -1) if raw_feats.ndim == 1 else raw_feats)
    return (t - t_mean) / t_std


def crowding_penalty(normed_feat, all_normed, radius, strength):
    """Sum of Gaussian penalties from all other individuals."""
    diffs = all_normed - normed_feat
    dists_sq = (diffs ** 2).sum(axis=1)
    penalties = np.exp(-dists_sq / (2 * radius ** 2))
    # Subtract self-interaction (distance 0 -> penalty 1)
    return strength * (penalties.sum() - 1.0)


def food_fitness(normed_feat, all_normed_pop):
    """Fitness = KDE density at location - crowding from population."""
    food = kde(normed_feat.reshape(-1, 1))[0]
    crowd = crowding_penalty(normed_feat, all_normed_pop, CROWDING_RADIUS, CROWDING_STRENGTH)
    return food - crowd


def random_level(grid_size, rng=random):
    n = grid_size
    flip = rng.random() < 0.5
    wall_prob = rng.uniform(0.15, 0.45)
    h_walls = [True if (r == 0 or r == n) else rng.random() < wall_prob
                for r in range(n + 1) for c in range(n)]
    v_walls = [True if (c == 0 or c == n) else rng.random() < wall_prob
                for r in range(n) for c in range(n + 1)]
    exit_side = rng.choice(["N", "S", "E", "W"])
    exit_pos = rng.randint(0, n - 1)

    occupied = set()
    def rand_pos():
        for _ in range(100):
            r, c = rng.randint(0, n - 1), rng.randint(0, n - 1)
            if (r, c) not in occupied:
                occupied.add((r, c))
                return (r, c)
        return (0, 0)

    player = rand_pos()
    mummy1 = rand_pos()
    mummy2 = rand_pos() if rng.random() < 0.4 else None
    scorpion = rand_pos() if rng.random() < 0.3 else None
    traps = []
    if rng.random() < 0.2: traps.append(rand_pos())
    if rng.random() < 0.1: traps.append(rand_pos())

    # Gate: col must be < n-1 (east edge must be interior)
    gate = None
    key = None
    if rng.random() < 0.15:
        gr = rng.randint(0, n - 1)
        gc = rng.randint(0, n - 2)  # col < n-1
        gate = (gr, gc)
        # Clear east wall at gate cell so gate is visible
        v_walls[gr * (n + 1) + gc + 1] = False
        key = rand_pos()

    return mummymaze_rust.Level.from_edges(
        grid_size, flip, h_walls, v_walls,
        exit_side, exit_pos, player, mummy1,
        mummy2, scorpion, traps, gate, key,
    )


def run_food_landscape_ga(
    grid_size=GRID_SIZE, pop_size=POP_SIZE, generations=GENERATIONS,
    tournament_k=TOURNAMENT_K, elite_frac=ELITE_FRAC, mutation_rate=MUTATION_RATE,
    crossover_modes=CROSSOVER_MODES, seed=SEED, snapshot_every=SNAPSHOT_EVERY,
):
    rng = random.Random(seed)

    # --- Generate initial population ---
    print("Generating initial population...")
    population = []  # list of (level, raw_features, normed_features)
    attempts = 0
    while len(population) < pop_size:
        batch = [random_level(grid_size, rng) for _ in range(pop_size * 3)]
        results = mummymaze_rust.ga_evaluate_batch(batch)
        for r in results:
            lev = r["level"]
            feat = level_features(r, lev)
            normed = to_normed(feat)[0]
            population.append((lev, feat, normed))
            if len(population) >= pop_size:
                break
        attempts += 1
        if attempts > 30:
            break

    print(f"  Initial population: {len(population)}")

    # --- Evolution loop ---
    history = []
    snapshots = []

    for gen in range(generations):
        # Compute fitness for entire population
        all_normed = np.array([ind[2] for ind in population])
        fitnesses = []
        for i, (lev, feat, normed) in enumerate(population):
            f = food_fitness(normed, all_normed)
            fitnesses.append(f)

        # Sort by fitness
        ranked = sorted(zip(fitnesses, population), key=lambda x: x[0], reverse=True)
        fitnesses_sorted = [f for f, _ in ranked]
        population = [ind for _, ind in ranked]

        best_fit = fitnesses_sorted[0]
        avg_fit = np.mean(fitnesses_sorted)
        history.append((gen, best_fit, avg_fit))

        # Snapshot for animation
        if gen % snapshot_every == 0 or gen == generations - 1:
            snap_normed = np.array([ind[2] for ind in population])
            snapshots.append((gen, snap_normed))

        if gen % 10 == 0 or gen == generations - 1:
            print(f"  Gen {gen:3d}: best={best_fit:.4f} avg={avg_fit:.4f}")

        # --- Selection + reproduction ---
        n_elite = max(1, int(pop_size * elite_frac))
        next_gen = list(population[:n_elite])

        offspring_levels = []
        while len(offspring_levels) < pop_size - n_elite:
            candidates = rng.sample(list(zip(fitnesses, population)),
                                     min(tournament_k, len(population)))
            _, parent = max(candidates, key=lambda x: x[0])

            if rng.random() < mutation_rate:
                child = mummymaze_rust.mutate(
                    parent[0], rng.randint(0, 2**63),
                    w_wall=W_WALL, w_move_entity=W_MOVE_ENTITY,
                    w_move_player=W_MOVE_PLAYER, w_move_exit=W_MOVE_EXIT,
                    w_add_entity=0.0, w_remove_entity=0.0,
                    extra_wall_prob=EXTRA_WALL_PROB,
                )
            else:
                candidates2 = rng.sample(list(zip(fitnesses, population)),
                                          min(tournament_k, len(population)))
                _, parent2 = max(candidates2, key=lambda x: x[0])
                mode = rng.choice(crossover_modes)
                child = mummymaze_rust.ga_crossover(
                    parent[0], parent2[0], mode, rng.randint(0, 2**63),
                )
            offspring_levels.append(child)

        # Evaluate offspring
        results = mummymaze_rust.ga_evaluate_batch(offspring_levels)
        for r in results:
            lev = r["level"]
            feat = level_features(r, lev)
            normed = to_normed(feat)[0]
            next_gen.append((lev, feat, normed))

        population = next_gen[:pop_size]

    print(f"\nFinal population: {len(population)} levels")
    return population, history, snapshots

In [ ]:
# Run it
population, history, snapshots = run_food_landscape_ga()

## 3. Fitness history

In [ ]:
gens, bests, avgs = zip(*history)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(gens, bests, label="Best", color="tab:blue")
ax.plot(gens, avgs, label="Avg", color="tab:orange")
ax.set_xlabel("Generation")
ax.set_ylabel("Fitness (food - crowding)")
ax.set_title("Food Landscape GA Fitness")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Animate population spreading across feature space

Shows 2D projections (BFS moves vs N states, BFS moves vs Log win prob) with the
KDE density as background contours. Population dots move each generation as
individuals compete for food.

In [ ]:
# Animate population over generations on 2D KDE slices
# We work in raw feature space for readable axes

proj_pairs = [(0, 1), (0, 2)]  # (BFS vs N states), (BFS vs Log win prob)
proj_names = [(FEATURE_NAMES[i], FEATURE_NAMES[j]) for i, j in proj_pairs]

# Precompute 2D KDE contours for each projection (from originals)
contour_data = []
for (i, j) in proj_pairs:
    data_2d = orig_features_raw[:, [i, j]].T
    kde_2d = gaussian_kde(data_2d)
    xi = np.linspace(orig_features_raw[:, i].min() - orig_features_raw[:, i].std() * 0.3,
                     orig_features_raw[:, i].max() + orig_features_raw[:, i].std() * 0.3, 60)
    yi = np.linspace(orig_features_raw[:, j].min() - orig_features_raw[:, j].std() * 0.3,
                     orig_features_raw[:, j].max() + orig_features_raw[:, j].std() * 0.3, 60)
    Xi, Yi = np.meshgrid(xi, yi)
    Zi = kde_2d(np.vstack([Xi.ravel(), Yi.ravel()])).reshape(Xi.shape)
    contour_data.append((Xi, Yi, Zi, xi, yi))

# Convert snapshots from normed space back to raw space
raw_snapshots = []
for gen, normed in snapshots:
    transformed = normed * t_std + t_mean
    raw = untransform_features(transformed)
    raw_snapshots.append((gen, raw))

fig, axes = plt.subplots(1, len(proj_pairs), figsize=(8 * len(proj_pairs), 7))
if len(proj_pairs) == 1:
    axes = [axes]

# Static elements
for idx, ((i, j), ax) in enumerate(zip(proj_pairs, axes)):
    Xi, Yi, Zi, _, _ = contour_data[idx]
    ax.contourf(Xi, Yi, Zi, levels=15, cmap="Blues", alpha=0.5)
    ax.scatter(orig_features_raw[:, i], orig_features_raw[:, j],
               s=2, alpha=0.08, color="gray")
    ax.set_xlabel(FEATURE_NAMES[i])
    ax.set_ylabel(FEATURE_NAMES[j])

scatters = []
for ax in axes:
    sc = ax.scatter([], [], s=25, color="tab:orange", edgecolors="black",
                     linewidths=0.5, alpha=0.8, zorder=5)
    scatters.append(sc)

title = fig.suptitle("", fontsize=13)

def update(frame_idx):
    gen, raw_feats = raw_snapshots[frame_idx]
    title.set_text(f"Generation {gen}")
    for idx, (i, j) in enumerate(proj_pairs):
        scatters[idx].set_offsets(raw_feats[:, [i, j]])
    return scatters + [title]

anim = FuncAnimation(fig, update, frames=len(raw_snapshots),
                      interval=300, repeat=True, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

## 5. Feature distribution comparison

In [ ]:
gen_feats = np.array([ind[1] for ind in population])

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for i, (ax, name) in enumerate(zip(axes, FEATURE_NAMES)):
    combined = np.concatenate([orig_features_raw[:, i], gen_feats[:, i]])
    bins = np.linspace(combined.min(), combined.max(), 31)
    ax.hist(orig_features_raw[:, i], bins=bins, alpha=0.5, label="Original", density=True, color="tab:blue")
    ax.hist(gen_feats[:, i], bins=bins, alpha=0.6, label="Generated", density=True, color="tab:orange")
    ax.set_xlabel(name)
    ax.set_ylabel("Density")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle(f"Feature Distributions: Original vs Food-Landscape GA (gs={GRID_SIZE}, {len(gen_feats)} levels)", fontsize=13)
plt.tight_layout()
plt.show()

## 6. Sample generated levels + BFS animation

In [ ]:
ENTITY_RADIUS = 0.3
TRAP_SIZE = 0.25

def draw_maze(ax, level_dict, frame=None, title=None):
    ax.clear()
    gs = level_dict["grid_size"]
    ax.set_xlim(-0.5, gs - 0.5)
    ax.set_ylim(gs - 0.5, -0.5)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_facecolor("#f5f0e0")

    for r in range(gs):
        for c in range(gs):
            ax.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1,
                         linewidth=0.5, edgecolor="#d0c8b0", facecolor="#f5f0e0"))

    h_walls, v_walls = level_dict["h_walls"], level_dict["v_walls"]
    for r in range(gs + 1):
        for c in range(gs):
            if h_walls[r * gs + c]:
                ax.plot([c - 0.5, c + 0.5], [r - 0.5, r - 0.5],
                        color="#2a1a0a", linewidth=3, solid_capstyle="round")
    for r in range(gs):
        for c in range(gs + 1):
            if v_walls[r * (gs + 1) + c]:
                ax.plot([c - 0.5, c - 0.5], [r - 0.5, r + 0.5],
                        color="#2a1a0a", linewidth=3, solid_capstyle="round")

    es, ep = level_dict["exit_side"], level_dict["exit_pos"]
    ec = "#4CAF50"
    if es == "N":   ax.plot([ep - 0.4, ep + 0.4], [-0.5, -0.5], color=ec, linewidth=4, solid_capstyle="round")
    elif es == "S": ax.plot([ep - 0.4, ep + 0.4], [gs - 0.5, gs - 0.5], color=ec, linewidth=4, solid_capstyle="round")
    elif es == "W": ax.plot([-0.5, -0.5], [ep - 0.4, ep + 0.4], color=ec, linewidth=4, solid_capstyle="round")
    elif es == "E": ax.plot([gs - 0.5, gs - 0.5], [ep - 0.4, ep + 0.4], color=ec, linewidth=4, solid_capstyle="round")

    if frame is not None:
        pr, pc = frame["player"]
        m1r, m1c, m1a = frame["mummy1"]
        m2r, m2c, m2a = frame["mummy2"]
        sr, sc, sa = frame["scorpion"]
        gate_open = frame["gate_open"]
    else:
        pr, pc = level_dict["player"]
        m1r, m1c, m1a = *level_dict["mummy1"], True
        m2 = level_dict["mummy2"]
        m2r, m2c, m2a = (*m2, True) if m2 else (0, 0, False)
        scorp = level_dict["scorpion"]
        sr, sc, sa = (*scorp, True) if scorp else (0, 0, False)
        gate_open = False

    for tr, tc in level_dict.get("traps", []):
        s = TRAP_SIZE
        ax.plot([tc-s, tc+s], [tr-s, tr+s], color="#8B0000", linewidth=2.5)
        ax.plot([tc-s, tc+s], [tr+s, tr-s], color="#8B0000", linewidth=2.5)
    if level_dict.get("key"):
        kr, kc = level_dict["key"]
        ax.plot(kc, kr, marker="P", markersize=12, color="#FFD700", markeredgecolor="#B8860B", markeredgewidth=1.5, zorder=5)
    if level_dict.get("gate"):
        gr, gc = level_dict["gate"]
        ax.plot([gc+0.5, gc+0.5], [gr-0.35, gr+0.35], color="#8B0000" if gate_open else "#90EE90", linewidth=4, solid_capstyle="round", zorder=6)

    mc = "#D32F2F" if level_dict["flip"] else "#F5F5DC"
    me = "#8B0000" if level_dict["flip"] else "#8B8B00"
    if m1a:
        ax.add_patch(plt.Circle((m1c, m1r), ENTITY_RADIUS, color=mc, ec=me, linewidth=2, zorder=8))
        ax.text(m1c, m1r, "M", ha="center", va="center", fontsize=9, fontweight="bold", color=me, zorder=9)
    if m2a:
        ax.add_patch(plt.Circle((m2c, m2r), ENTITY_RADIUS, color=mc, ec=me, linewidth=2, zorder=8))
        ax.text(m2c, m2r, "M", ha="center", va="center", fontsize=9, fontweight="bold", color=me, zorder=9)
    if sa:
        ax.add_patch(plt.Circle((sc, sr), ENTITY_RADIUS, color="#FF8C00", ec="#8B4500", linewidth=2, zorder=8))
        ax.text(sc, sr, "S", ha="center", va="center", fontsize=9, fontweight="bold", color="#8B4500", zorder=9)

    result = frame["result"] if frame else "ok"
    pc_color = "#4FC3F7" if result == "ok" else ("#4CAF50" if result == "win" else "#EF5350")
    ax.add_patch(plt.Circle((pc, pr), ENTITY_RADIUS, color=pc_color, ec="#01579B", linewidth=2, zorder=10))
    ax.text(pc, pr, "P", ha="center", va="center", fontsize=9, fontweight="bold", color="#01579B", zorder=11)
    if title: ax.set_title(title, fontsize=11)


def animate_solution(level, interval=500, figsize=(5, 5)):
    actions = mummymaze_rust.solve_actions(level)
    if actions is None:
        print("Unsolvable!"); return None
    frames = mummymaze_rust.replay_actions(level, actions)
    level_dict = level.to_dict()
    action_names = ["N", "S", "E", "W", "Wait"]
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    def update(i):
        t = f"Step 0/{len(actions)} (initial)" if i == 0 else f"Step {i}/{len(actions)} (action: {action_names[actions[i-1]]})"
        draw_maze(ax, level_dict, frames[i], title=t)
    anim = FuncAnimation(fig, update, frames=len(frames), interval=interval, repeat=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())


# Show 8 generated levels
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flatten()):
    if i < len(population):
        lev, feat, normed = population[i]
        bfs = int(feat[0])
        ns = int(feat[1])
        draw_maze(ax, lev.to_dict(), title=f"#{i+1} bfs={bfs} states={ns}")
    else:
        ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Animate BFS solution of first generated level
animate_solution(population[0][0], interval=600)